# 🛰️ Avatar Renderer — Temporary **Colab GPU Job Server**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ruslanmv/avatar-renderer-mcp/blob/main/colab/Avatar_Renderer_GPU_Server.ipynb)

Turns this Colab GPU runtime into a small, **token-authenticated** HTTP server that
your local **Claude Code** can call to run GPU tests + renders on a branch — without
SSH and without an arbitrary-shell endpoint. See `docs/COLAB_GPU_TESTING.md`.

```
Local Claude Code  --curl-->  Cloudflare Quick Tunnel  -->  Colab FastAPI (allowlisted)
                                                              ├─ /git/pull  /setup
                                                              ├─ /gpu/check /engines
                                                              ├─ /render/sample (orchestrate)
                                                              └─ /jobs/{id}  /artifact/{name}
```

**First:** `Runtime → Change runtime type → GPU` (T4 is fine; A100/L4 for LatentSync).

## 1 · GPU + clone repo

In [ ]:
!nvidia-smi || echo 'No GPU! Runtime -> Change runtime type -> GPU'
import os, sys, subprocess, time, re, secrets
from pathlib import Path

BRANCH = 'claude/project-review-9nn71'  #@param {type:'string'}
REPO_DIR = Path('/content/work/avatar-renderer-mcp')
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
if not REPO_DIR.exists():
    !git clone -q https://github.com/ruslanmv/avatar-renderer-mcp.git {REPO_DIR}
!cd {REPO_DIR} && git fetch -q --all && git checkout -q {BRANCH} && git pull -q origin {BRANCH} || true
os.chdir(REPO_DIR); sys.path.insert(0, str(REPO_DIR))
print('repo:', REPO_DIR, '| branch:', subprocess.run(['git','rev-parse','--abbrev-ref','HEAD'],capture_output=True,text=True).stdout.strip())

## 2 · Install server + lip-sync dependencies

`fastapi`/`uvicorn`/`cloudflared` for the server + tunnel; the rest are the render deps.
We don't touch Colab's pre-installed CUDA torch.

In [ ]:
!apt-get -qq install -y ffmpeg > /dev/null
!pip install -q fastapi 'uvicorn[standard]' python-multipart pydantic edge-tts \
    'librosa>=0.10' soundfile opencv-python-headless imageio imageio-ffmpeg \
    huggingface_hub gfpgan basicsr facexlib 2>/dev/null || true
# Cloudflare quick tunnel (no login).
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('deps ready')

## 3 · Token + environment

A fresh random token each session. The server reads these env vars.

In [ ]:
TOKEN = secrets.token_urlsafe(32)
os.environ['COLAB_GPU_TOKEN'] = TOKEN
os.environ['REPO_DIR']     = str(REPO_DIR)
os.environ['ARTIFACT_DIR'] = '/content/artifacts'
os.environ['INPUT_DIR']    = '/content/inputs'
os.environ['MODEL_ROOT']   = str(REPO_DIR / 'models')
os.environ['EXT_DEPS_DIR'] = str(REPO_DIR / 'external_deps')
Path('/content/artifacts').mkdir(exist_ok=True); Path('/content/inputs').mkdir(exist_ok=True)
print('TOKEN set (kept secret; printed with the URL below)')

## 4 · (Optional) Pre-install engines now

You can install engines here, or later via the `/setup` endpoint from Claude Code.
On a **T4** use `musetalk diff2lip`; add `latentsync` only on A100/L4.

In [ ]:
PREINSTALL = True  #@param {type:'boolean'}
ENGINES = 'musetalk diff2lip'  #@param {type:'string'}
if PREINSTALL:
    !bash scripts/colab_gpu_setup.sh 1 {ENGINES} 2>&1 | tail -40
else:
    print('Skipped — call POST /setup from Claude Code instead.')

## 5 · Start the server

Force-frees port 8080 (kills any zombie uvicorn from a previous run), imports the
app first (so errors are visible), then runs uvicorn logging to `/content/uvicorn.log`.
Safe to re-run as many times as you like.

In [ ]:
import importlib, urllib.request, signal

# 1) Evict anything squatting on :8080 (a leftover server from an earlier run
#    would otherwise keep serving STALE code and the new one fails to bind).
try: server.terminate()
except Exception: pass
!fuser -k 8080/tcp 2>/dev/null || true
!pkill -f 'uvicorn.*colab.colab_gpu_server' 2>/dev/null || true
time.sleep(2)

# 2) Surface import errors directly (uvicorn would otherwise exit silently).
sys.path.insert(0, str(REPO_DIR))
import colab.colab_gpu_server as _s; importlib.reload(_s)
print('app routes:', sorted({r.path for r in _s.app.routes}))

# 3) Start uvicorn, logging to a readable file.
logf = open('/content/uvicorn.log', 'w')
server = subprocess.Popen(
    [sys.executable,'-m','uvicorn','colab.colab_gpu_server:app','--host','0.0.0.0','--port','8080'],
    cwd=str(REPO_DIR), env=os.environ.copy(), stdout=logf, stderr=subprocess.STDOUT)
time.sleep(7)

# 4) Verify — and prove it's OUR server (the /ping JSON), not a zombie.
try:
    r = urllib.request.urlopen('http://localhost:8080/ping', timeout=5).read().decode()
    print('/ping ->', r)
    assert 'avatar-renderer-colab-gpu' in r, 'A different server answered :8080 — re-run this cell.'
    print('✅ server is up')
except Exception as e:
    print('server not up:', e, '\n--- uvicorn.log ---\n', open('/content/uvicorn.log').read()[-3000:])

## 6 · Open the tunnel → get the public URL

In [ ]:
tunnel = subprocess.Popen(['cloudflared','tunnel','--url','http://localhost:8080'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
PUBLIC_URL = None
for _ in range(90):
    line = tunnel.stdout.readline()
    if not line: break
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m: PUBLIC_URL = m.group(0); break
print('='*64)
print('PUBLIC_URL :', PUBLIC_URL)
print('TOKEN      :', TOKEN)
print('='*64)
print('\nPaste these into your LOCAL terminal where Claude Code runs:\n')
print(f"export COLAB_GPU_URL='{PUBLIC_URL}'")
print(f"export COLAB_GPU_TOKEN='{TOKEN}'")

## 7 · Use it from local Claude Code

In your local repo terminal (after the two `export`s above):

```bash
scripts/claude_colab_client.sh GET  /health
scripts/claude_colab_client.sh GET  /engines
scripts/claude_colab_client.sh POST /git/pull '{"branch":"claude/project-review-9nn71"}'
# render on a chosen engine, then wait + download:
JID=$(scripts/claude_colab_client.sh POST /render/sample \
  '{"engine":"musetalk","quality_mode":"high_quality","text":"Hello from Colab GPU"}' \
  | python3 -c 'import sys,json;print(json.load(sys.stdin)["job_id"])')
scripts/claude_colab_client.sh wait $JID
scripts/claude_colab_client.sh DOWNLOAD /artifact/<name-from-job> out.mp4
```

Premium tiers never silently downgrade: if an engine isn't installed, the job
errors (check `/jobs/<id>` log) — run `POST /setup` for that engine and retry.

## 8 · Stop (when finished — keep sessions short)

In [ ]:
try: tunnel.terminate()
except Exception: pass
try: server.terminate()
except Exception: pass
!fuser -k 8080/tcp 2>/dev/null || true
!pkill -f cloudflared 2>/dev/null || true
print('tunnel + server stopped (port 8080 freed). You can also disconnect the runtime.')